# 03 - Baseline Model Eğitimi
Heuristic skorlardan bootstrap edilen baseline modellerin eğitimi ve karşılaştırması.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from src.data_ingestion import load_hazard_data, load_building_data
from src.data_processing import clean_building_data, merge_hazard_building
from src.feature_engineering import prepare_training_data, get_feature_names

## 1. Veri ve Feature Hazırlama

In [ ]:
hazard_df = load_hazard_data()
building_df = load_building_data()
building_df = clean_building_data(building_df)
merged_df = merge_hazard_building(hazard_df, building_df)

X, y = prepare_training_data(merged_df)
base_features = [f for f in get_feature_names() if f in X.columns]
X_model = X[base_features]

print(f'Veri boyutu: {X_model.shape}')
print(f'Sınıf dağılımı: {dict(y.value_counts())}')

## 2. Model Eğitimi

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y, test_size=0.25, random_state=42
)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print('Logistic Regression:')
print(f'  Accuracy: {accuracy_score(y_test, lr_pred):.3f}')
print(f'  F1: {f1_score(y_test, lr_pred, average="weighted", zero_division=0):.3f}')

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print('\nRandom Forest:')
print(f'  Accuracy: {accuracy_score(y_test, rf_pred):.3f}')
print(f'  F1: {f1_score(y_test, rf_pred, average="weighted", zero_division=0):.3f}')

## 3. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(rf.feature_importances_, index=base_features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('Random Forest Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Notlar
- Bu baseline modeller heuristic skorlardan bootstrap edilmiştir.
- Gerçek etiketli veri ile yeniden eğitim yapılmalıdır.
- Veri sayısı çok az olduğundan metrikler yanıltıcı olabilir.